# NYC 311 AutoDDG: milestone notebook

This notebook is the presentation of the project. Run it from top to bottom. Every milestone ends with an **Outcome** section: the table or file that goes into the report.

**Question.** How well can an AutoDDG-style description, grounded in an Apache Spark profile, describe NYC 311 service requests and make that dataset easier to find than the short portal text?

**Data.**
- Fact table: [311 Service Requests from 2020 to Present](https://data.cityofnewyork.us/d/erm2-nwe9), id `erm2-nwe9`.
- Dimension table: [ZIP Code Tabulation Areas](https://data.cityofnewyork.us/d/35j5-n34v), id `35j5-n34v` (221 rows).

The default run is a **dated sample**, not the full archive. The profile records the window, and the generated text is required to say so.


## Before you run anything

1. Use **Python 3.11** and **JDK 17**. PySpark will not start on a machine with no `java` on `PATH`.
2. From the repo root: `python3.11 -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt`.
3. `cp .env.example .env` and set `OPENAI_API_KEY`. Milestones 1, 2, and the human BM25 index run without it. Description generation and the judge do not.
4. Leave the default window (`2025-06-01` inclusive to `2025-06-03` exclusive, at most 20,000 rows) until this notebook finishes once.
5. Start Jupyter with the repo root as the working directory, or start it from `notebooks/`. The setup cell handles both.

Do not commit `.env`, the CSV pages, or the Parquet output.


## How these steps map onto the proposal

The proposal's four milestones are the grading structure. This notebook uses the same four numbers, with the concrete 311 pipeline inside them.

| Proposal milestone | What you finish in this notebook | What you still add before the deadline |
| --- | --- | --- |
| 1. Setup, baselines, Spark | Data lake, portal catalog, raw audit | One Auctus dataset that already has a human description |
| 2. Generation and Spark patterns | ZIP/ZCTA broadcast join, resolution-time pattern, SFD and UFD | Run AutoDDG's own profiler if the course repo is required, using this Spark profile as the check |
| 3. Evaluation | BM25 MRR / Recall@1 and the LLM judge | One NYC or Auctus dataset that has **no** description, plus a second judge only if you have time |
| 4. Report, due 9 December | The outcome files below, and the section map at the end | ACM-style paper with those numbers and a link to the GitHub repo |

Stretch work from the proposal (multimodal data, TabLib, Zenodo) stays out of this notebook until the table above is done.


## Setup check


In [1]:
import shutil
import sys
from pathlib import Path

import os

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display

from src import config


def show_table(title, rows, columns=None, limit=20):
    display(Markdown(f"### {title}"))
    rows = list(rows)[:limit]
    if not rows:
        display(Markdown("_No rows._"))
        return
    columns = columns or list(rows[0].keys())
    lines = [
        "| " + " | ".join(columns) + " |",
        "| " + " | ".join("---" for _ in columns) + " |",
    ]
    for row in rows:
        cells = [str(row.get(column, "")).replace("|", "/") for column in columns]
        lines.append("| " + " | ".join(cells) + " |")
    display(Markdown("\n".join(lines)))


java = shutil.which("java")
display(Markdown(
    f"""### Environment
| Item | Value |
| --- | --- |
| Repo | `{config.ROOT}` |
| Sample window | `{config.START_DATE}` inclusive, `{config.END_DATE}` exclusive |
| Row cap | `{config.SAMPLE_ROWS}` (`0` means the whole window) |
| Java on PATH | `{java or "MISSING — install Temurin 17 before Milestone 1 Spark"}` |
| OpenAI key loaded | `{bool(os.getenv("OPENAI_API_KEY"))}` |
| Model | `{config.OPENAI_MODEL}` |
"""
))


### Environment
| Item | Value |
| --- | --- |
| Repo | `/Users/lvcaitong/Desktop/NYU/Big data/Big Data final` |
| Sample window | `2025-06-01` inclusive, `2025-06-03` exclusive |
| Row cap | `20000` (`0` means the whole window) |
| Java on PATH | `/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home/bin/java` |
| OpenAI key loaded | `False` |
| Model | `gpt-4o` |


## Milestone 1 — Ingestion, storage, and raw profiling

**Goal.** Land a reproducible slice of both tables and record the schema, key uniqueness, and missingness that later stages are allowed to talk about.

**Do these steps**

1. Create the lake directories (the download functions do this).
2. Download 311 from the SODA CSV endpoint in pages of at most 50,000 rows. A single `$limit=500000` request is not a reliable way to get half a million rows. The code orders by `unique_key` inside the date window so the same `.env` reproduces the same pages.
3. Download ZCTA centroids only. Skip `the_geom`. The dimension table is the broadcast side, and the polygon column is not used by the ZIP equality join.
4. Save the official portal name and description for 311 and four other datasets. Those descriptions are the human baseline. Fetch them here, before any model text exists.
5. Start Spark locally. Read every column as a string. Inferring types would turn `incident_zip` into an integer and drop leading zeros.
6. Count rows, distinct `unique_key`, and null or blank values on the join and category fields. Classify columns as identifier, temporal, spatial, text, or categorical.

**Outcome files:** `data/raw/fact_311/`, `data/raw/dim_zcta/`, `data/processed/catalog.json`, `data/processed/m1_raw_audit.json`.


In [2]:
from src.ingest import download_fact_311, download_zcta, fetch_catalog

fact_manifest = download_fact_311()
zcta_manifest = download_zcta()
catalog = fetch_catalog()

show_table(
    "Download manifest",
    [
        {"table": "311 fact", **{k: fact_manifest[k] for k in ("dataset_id", "rows", "pages", "where")}},
        {"table": "ZCTA dimension", **{k: zcta_manifest[k] for k in ("dataset_id", "rows", "pages", "select")}},
    ],
)
show_table(
    "Portal catalog (human baselines)",
    [
        {
            "id": row["id"],
            "name": row["name"],
            "columns": row["n_columns"],
            "description": (row["human_description"][:180] + "…") if len(row["human_description"]) > 180 else row["human_description"],
        }
        for row in catalog
    ],
)


### Download manifest

| table | dataset_id | rows | pages | where |
| --- | --- | --- | --- | --- |
| 311 fact | erm2-nwe9 | 19522 | 1 | created_date >= '2025-06-01T00:00:00.000' AND created_date < '2025-06-03T00:00:00.000' |
| ZCTA dimension | 35j5-n34v | 221 | 1 |  |

### Portal catalog (human baselines)

| id | name | columns | description |
| --- | --- | --- | --- |
| erm2-nwe9 | 311 Service Requests from 2020 to Present | 48 | NOTE: Learn more about the latest changes to this dataset: https://opendata.cityofnewyork.us/311-service-requests-from-2010-to-present-updates/ 311 responds to thousands of inquiri… |
| h9gi-nx95 | Motor Vehicle Collisions - Crashes | 29 | The Motor Vehicle Collisions crash table contains details on the crash event. Each row represents a crash event. The Motor Vehicle Collisions data tables contain information from a… |
| 43nn-pn8j | DOHMH New York City Restaurant Inspection Results | 31 | The dataset contains every sustained or not yet adjudicated violation citation from every full or special program inspection conducted up to three years prior to the most recent in… |
| 5uac-w243 | NYPD Complaint Data Current (Year To Date) | 41 | This dataset includes all valid felony, misdemeanor, and violation crimes reported to the New York City Police Department (NYPD) for all complete quarters so far this year (2019). … |
| 35j5-n34v | ZIP Code Tabulation Areas | 8 | This is a subset of ZIP Code Tabulation Areas or ZCTAs (pronounced zik-tahs) that fall within New York City. ZCTAs are a geographic product of the U.S. Census Bureau created to all… |

### Spark load and structural audit

If the next cell says Java is missing, install it, export `JAVA_HOME`, restart the kernel, and run from the setup cell again. A Spark session that was created earlier in the same kernel keeps its old configuration, so restart rather than calling `getOrCreate` a second time after changing memory settings.


In [3]:
if shutil.which("java") is None:
    raise RuntimeError(
        "Java is not on PATH. Install Temurin 17, export JAVA_HOME, restart the kernel, and rerun from the setup cell."
    )

from src.spark_etl import audit_raw, load_fact, load_zcta, start_spark

spark = start_spark()
fact_raw = load_fact(spark)
zcta_raw = load_zcta(spark)
audit = audit_raw(fact_raw)

display(Markdown(
    f"""### Outcome — raw 311 audit
| Check | Value |
| --- | --- |
| Rows in this sample | {audit['total_rows']:,} |
| Distinct `unique_key` | {audit['unique_keys']:,} |
| Duplicate keys | {audit['duplicate_keys']:,} |
| Columns | {audit['n_columns']} |
| ZCTA rows loaded | {zcta_raw.count():,} |
| Window | {audit['sample_window']['start']} to {audit['sample_window']['end']} (end exclusive) |
"""
))
show_table(
    "Null or blank counts on fields the join and the description depend on",
    [{"column": name, "null_or_blank": n} for name, n in audit["null_or_blank_counts"].items()],
)
roles = {}
for column in audit["columns"]:
    roles.setdefault(column["role"], []).append(column["name"])
show_table(
    "Column roles",
    [{"role": role, "columns": ", ".join(names)} for role, names in sorted(roles.items())],
)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 11:30:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Outcome — raw 311 audit
| Check | Value |
| --- | --- |
| Rows in this sample | 19,522 |
| Distinct `unique_key` | 19,522 |
| Duplicate keys | 0 |
| Columns | 44 |
| ZCTA rows loaded | 221 |
| Window | 2025-06-01 to 2025-06-03 (end exclusive) |


### Null or blank counts on fields the join and the description depend on

| column | null_or_blank |
| --- | --- |
| unique_key | 0 |
| created_date | 0 |
| closed_date | 229 |
| complaint_type | 0 |
| descriptor | 0 |
| incident_zip | 176 |
| borough | 0 |
| latitude | 331 |
| longitude | 331 |
| status | 0 |

### Column roles

| role | columns |
| --- | --- |
| categorical | agency, agency_name, complaint_type, location_type, street_name, cross_street_1, cross_street_2, intersection_street_1, intersection_street_2, address_type, landmark, facility_type, status, community_board, council_district, police_precinct, open_data_channel_type, park_facility_name, park_borough, vehicle_type, taxi_company_borough, taxi_pick_up_location, bridge_highway_name, bridge_highway_direction, road_ramp, bridge_highway_segment |
| identifier | unique_key, bbl |
| spatial | incident_zip, city, borough, x_coordinate_state_plane, y_coordinate_state_plane, latitude, longitude, location |
| temporal | created_date, closed_date, due_date, resolution_action_updated_date |
| text | descriptor, descriptor_2, incident_address, resolution_description |

### Milestone 1 is done when

- The fact manifest row count is greater than zero and not larger than `SAMPLE_ROWS`.
- `duplicate_keys` is zero. Investigate before joining if it is not.
- `incident_zip`, `created_date`, and `complaint_type` show their real missingness. Those counts are ground truth for the judge later.
- The catalog has a non-empty human description for every id that the search queries name. If a portal id was retired, remove that id from `CATALOG_IDS` and `SEARCH_QUERIES` in `src/config.py` together, and write down the change in the report.


## Milestone 2 — Cleaning, broadcast join, partitioned Parquet

**Goal.** Produce one curated table: a 311 row with a 5-digit ZIP, a real timestamp, and ZCTA centroid columns when the ZIP matches a tabulation area.

**Do these steps**

1. Build `clean_zip` from the digits in `incident_zip`. Keep the first five. ZIP+4 values such as `10001-1234` become `10001`. Do not zero-pad an empty value into `00000`.
2. Parse `created_date` and `closed_date` with both timestamp styles. The SODA file is `yyyy-MM-dd'T'HH:mm:ss.SSS`. A bulk portal export is `MM/dd/yyyy hh:mm:ss a`. Using only the second pattern silently nulls every SODA timestamp.
3. Drop rows whose ZIP is not five digits or whose created timestamp did not parse. Record how many rows each rule removes. That drop is part of the result, not a hidden filter.
4. Broadcast the ZCTA table. It is 221 rows, well under a megabyte, so Spark can replicate it instead of shuffling the fact table.
5. Left-join on `clean_zip = dim_zip`. Report the match rate and the most common unmatched ZIPs.
6. For the top complaint types, compute the median hours from `created_timestamp` to `closed_timestamp`. This is the Spark pattern the description is allowed to mention.
7. Write Snappy Parquet partitioned by `year` and `month` to `data/curated/nyc_311_zcta_enriched`, then read it back and count rows per partition.

**Interpret the match rate in the report.** `incident_zip` is a USPS ZIP. `zcta5` is a Census ZCTA. They often agree in New York City and they are not the same geography, so a match rate below 100% is the honest result.


In [4]:
from src.spark_etl import (
    apply_fact_filters,
    broadcast_join,
    clean_fact,
    clean_zcta,
    join_report,
    partition_counts,
    resolution_pattern,
    write_curated,
)

fact_clean = clean_fact(fact_raw)
fact_kept, cleaning = apply_fact_filters(fact_clean)
zcta_clean = clean_zcta(zcta_raw)
joined = broadcast_join(fact_kept, zcta_clean)
joined.cache()
match = join_report(joined)
resolution = resolution_pattern(joined)
curated_path = write_curated(joined)
partitions = partition_counts(spark)

show_table("Outcome — rows removed by cleaning", [cleaning])
display(Markdown(
    f"""### Outcome — broadcast join
| Check | Value |
| --- | --- |
| Rows after cleaning | {match['rows']:,} |
| Matched to a ZCTA | {match['matched_rows']:,} |
| Unmatched | {match['unmatched_rows']:,} |
| Match rate | {match['match_rate']:.1%} |
| Curated Parquet | `{curated_path}` |
"""
))
display(Markdown(match["note"]))
show_table("Most common ZIPs with no ZCTA match", match["top_unmatched_zips"])
show_table("Outcome — median hours to close, top complaint types", resolution)
show_table("Outcome — Parquet partitions", partitions)


26/11/09 11:30:53 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


### Outcome — rows removed by cleaning

| rows_before_filter | rows_with_valid_zip | rows_dropped_bad_zip | rows_dropped_bad_timestamp_after_zip | rows_kept |
| --- | --- | --- | --- | --- |
| 19522 | 19346 | 176 | 0 | 19346 |

### Outcome — broadcast join
| Check | Value |
| --- | --- |
| Rows after cleaning | 19,346 |
| Matched to a ZCTA | 19,334 |
| Unmatched | 12 |
| Match rate | 99.9% |
| Curated Parquet | `/Users/lvcaitong/Desktop/NYU/Big data/Big Data final/data/curated/nyc_311_zcta_enriched` |


incident_zip is a USPS ZIP and zcta5 is a Census ZCTA. Equality is an approximation, so a match rate below 1 is expected.

### Most common ZIPs with no ZCTA match

| clean_zip | rows |
| --- | --- |
| 10121 | 4 |
| 10118 | 3 |
| 21044 | 1 |
| 07032 | 1 |
| 10281 | 1 |
| 10000 | 1 |
| 10048 | 1 |

### Outcome — median hours to close, top complaint types

| complaint_type | n | median_resolution_hours |
| --- | --- | --- |
| Illegal Parking | 3329 | 1.2 |
| Noise - Residential | 2427 | 0.71 |
| Noise - Street/Sidewalk | 1682 | 0.58 |
| Blocked Driveway | 862 | 1.17 |
| UNSANITARY CONDITION | 650 | 208.8 |
| Noise - Commercial | 571 | 0.51 |
| Street Condition | 417 | 26.21 |
| Noise - Vehicle | 409 | 0.73 |
| Dirty Condition | 402 | 21.34 |
| HEAT/HOT WATER | 360 | 32.68 |

### Outcome — Parquet partitions

| year | month | rows |
| --- | --- | --- |
| 2025 | 6 | 19346 |

### Milestone 2 is done when

- `rows_kept` is explained by the two drop counts.
- The match rate is printed, and the unmatched ZIP list is saved in `data/processed/m2_join_report.json`.
- Reading the Parquet path returns the same row total as `match['rows']`.
- `median_resolution_hours` is null only when `closed_date` did not parse. Do not fill it with a guess.


## Milestone 3 — Spark profile and AutoDDG descriptions

**Goal.** Give the model a profile, not the raw table. The profile is the contract that blocks invented counts.

**Do these steps**

1. From the joined DataFrame, compute row count, column names and roles, null counts, approximate distinct counts, latitude/longitude bounds, top complaint types, top boroughs, top agencies, the join match rate, and the resolution-time pattern.
2. Set `is_sample` from the row cap. The prompt forbids the model from describing this slice as the full 2020–present archive.
3. Save `data/processed/content_profile.json`. Open it and check that every number you would want in the paper is already there. If it is not in the file, do not ask the model to say it.
4. Call the model once. Ask for JSON with two fields:
   - **SFD** (search-focused): dense text with dataset ids, column names, join keys, and the categories that appear in the profile.
   - **UFD** (user-focused): two paragraphs on what a row is, the time window, the join, and one use.
5. Save both the JSON and the plain-text version. Temperature stays at 0.2 so a rerun does not wander.

The next cell fails with a clear message when `OPENAI_API_KEY` is unset. The profile file is still written.


In [5]:
from src.profile import build_profile
from src.autoddg_engine import generate_descriptions

profile = build_profile(
    joined,
    match,
    resolution,
    is_sample=config.SAMPLE_ROWS != 0,
)
display(Markdown(
    f"""### Outcome — profile the model will see
| Field | Value |
| --- | --- |
| Dataset | {profile['dataset_name']} |
| Rows | {profile['total_rows']:,} |
| Sample | {profile['is_sample']} |
| Window | {profile['sample_window']['start_inclusive']} to {profile['sample_window']['end_exclusive']} |
| Join match rate | {profile['join']['match_rate']:.1%} |
| Columns described | {len(profile['columns'])} |
| Saved | `data/processed/content_profile.json` |
"""
))
show_table("Top complaint types in the profile", profile["top_complaint_types"])


### Outcome — profile the model will see
| Field | Value |
| --- | --- |
| Dataset | NYC 311 Service Requests enriched with ZCTA centroids |
| Rows | 19,346 |
| Sample | True |
| Window | 2025-06-01 to 2025-06-03 |
| Join match rate | 99.9% |
| Columns described | 56 |
| Saved | `data/processed/content_profile.json` |


### Top complaint types in the profile

| value | count | share |
| --- | --- | --- |
| Illegal Parking | 3329 | 0.1721 |
| Noise - Residential | 2427 | 0.1255 |
| Noise - Street/Sidewalk | 1682 | 0.0869 |
| Blocked Driveway | 862 | 0.0446 |
| UNSANITARY CONDITION | 650 | 0.0336 |
| Noise - Commercial | 571 | 0.0295 |
| Street Condition | 451 | 0.0233 |
| Noise - Vehicle | 409 | 0.0211 |
| Dirty Condition | 409 | 0.0211 |
| HEAT/HOT WATER | 362 | 0.0187 |

In [6]:
descriptions = generate_descriptions(profile)
display(Markdown("### Outcome — search-focused description (SFD)\n\n" + descriptions["sfd"]))
display(Markdown("### Outcome — user-focused description (UFD)\n\n" + descriptions["ufd"]))
display(Markdown("Saved to `data/processed/autoddg_descriptions.json`."))


RuntimeError: OPENAI_API_KEY is missing. Copy .env.example to .env, set the key, and restart the kernel.

### Milestone 3 is done when

- You can point at one number in the SFD and find that same number in `content_profile.json`.
- The UFD names the sample window.
- Neither text mentions a borough, agency, or complaint type that is absent from the top-k lists.
- You have not edited the model text by hand. If you change the prompt, change it in `src/autoddg_engine.py` and rerun the cell so the paper matches the code.


## Milestone 4 — Retrieval, LLM judge, and the report

**Goal.** Show whether the SFD is easier to retrieve than the portal description, and whether the text stays inside the Spark profile.

### Step 1. BM25 on a corpus, not on two scores

Do not index one short human sentence and one long SFD and then compare the two BM25 scores. A longer keyword list almost always scores higher, and two documents are not a search task.

The implemented comparison:

1. One document per catalog dataset. The human index uses the official portal name plus description.
2. Queries live in `src/config.py`. They were written from the dataset titles, not from model output. Do not edit them after you read the SFD. If you must change one, say so in the report.
3. Report **MRR** and **Recall@1**, plus the rank of the correct dataset for each query.
4. Build a second index that replaces **only** the 311 document with the SFD. Leave every other document as portal text.
5. Compare MRR, not raw scores. Say in the paper that five datasets is a pilot. Add more catalog ids before you claim a general result.

The human index runs even if generation was skipped. The SFD index appears after Milestone 3.


In [ ]:
from src.search_eval import evaluate_retrieval

retrieval = evaluate_retrieval()
human = retrieval["human_portal_index"]
sfd_index = retrieval["autoddg_sfd_index"]
display(Markdown(
    f"""### Outcome — retrieval
| Index | Documents | Queries | MRR | Recall@1 |
| --- | --- | --- | --- | --- |
| Portal descriptions | {human['n_documents']} | {human['n_queries']} | {human['mrr']} | {human['recall_at_1']} |
| 311 document replaced by SFD | {sfd_index['n_documents'] if sfd_index else "—"} | {sfd_index['n_queries'] if sfd_index else "—"} | {sfd_index['mrr'] if sfd_index else "run Milestone 3"} | {sfd_index['recall_at_1'] if sfd_index else "run Milestone 3"} |
"""
))
show_table(
    "Rank of the correct dataset under portal text",
    human["queries"],
    ["query", "relevant_id", "rank", "top_id"],
)
if sfd_index:
    show_table(
        "Rank of the correct dataset after the 311 SFD swap",
        sfd_index["queries"],
        ["query", "relevant_id", "rank", "top_id"],
    )
display(Markdown("Saved to `data/processed/eval_bm25.json`."))


### Step 2. LLM-as-a-judge

Score the generated SFD and UFD against `content_profile.json` only. Temperature is 0. The rubric is faithfulness, completeness, and searchability, each from 1 to 5, plus a list of unsupported claims. A high searchability score does not repair an unsupported claim. Quote the unsupported-claims list in the report if it is non-empty, and fix the prompt rather than the prose.


In [ ]:
from src.autoddg_engine import judge_description

judgment = judge_description(profile, descriptions)
scores = judgment["scores"]
display(Markdown(
    f"""### Outcome — judge
| Rubric | Score |
| --- | --- |
| Faithfulness | {scores.get("faithfulness")} |
| Completeness | {scores.get("completeness")} |
| Searchability | {scores.get("searchability")} |
"""
))
show_table(
    "Unsupported claims",
    [{"claim": item} for item in scores.get("unsupported_claims") or []],
)
show_table(
    "Missing elements",
    [{"element": item} for item in scores.get("missing_elements") or []],
)
display(Markdown(scores.get("rationale") or ""))
display(Markdown("Saved to `data/processed/eval_judge.json`."))


### Step 3. What else the proposal still requires

Do this after the outcome files above exist.

1. **Auctus baseline.** Pick one dataset on [Auctus](https://auctus.vida-nyu.org/) that already has a human description. Download a small CSV, add its id, name, and description to the catalog JSON, and add one frozen query whose relevant id is that dataset. Rerun the BM25 cell.
2. **Missing dictionary.** Pick one NYC Open Data or Auctus dataset whose description is empty or a single sentence. Profile a sample in Spark and generate an SFD/UFD with the same prompt. That is the "no prior data dictionary" case in the proposal. Keep it separate from the 311 numbers.
3. **Scale.** When the sample looks right, set `START_DATE=2020-01-01`, choose an end date, set `SAMPLE_ROWS=0`, and rerun. Expect tens of millions of rows and a multi-gigabyte Parquet tree. Raise `SPARK_DRIVER_MEMORY` and `SPARK_SHUFFLE_PARTITIONS` first. Point the report at whichever window you actually ran.
4. **Second judge and multimodal work.** Only after the items above. The proposal marks images, video, and TabLib as stretch goals.

### Step 4. Write the paper from the outcome files

Use the ACM template if you want LaTeX. The required sections, and the file that fills each one:

| Section | Use |
| --- | --- |
| Introduction | The discovery problem and the 311 question. Link the GitHub repo. |
| Problem formulation | Sample window, `clean_zip` = `dim_zip`, SFD vs UFD, MRR, Recall@1, three judge scores. |
| Related work | AutoDDG, Spark profiling, BM25, LLM-as-a-judge. Cite every paper you name. |
| Methods, architecture, and design | Lake layout, paged SODA download, string schema, broadcast join, profile-only prompting, frozen queries. |
| Results | The outcome tables in this notebook. State the match-rate caveat and that the corpus is a pilot until you add datasets. |
| References | Portal dataset pages plus the papers. |

### Step 5. Make the repo reproducible

The report link is <https://github.com/tl4732-cyber/Big-Data_Dataset-Description-Generation>.

From the repo root, after you are ready to publish:

```bash
git init
git add .
git commit -m "Add the reproducible AutoDDG Spark pipeline and milestone notebook."
git branch -M main
git remote add origin https://github.com/tl4732-cyber/Big-Data_Dataset-Description-Generation.git
git push -u origin main
```

Confirm the push does not contain `.env` or the raw CSV. `README.md` is the reproduction guide a grader should be able to follow without this chat.
